# Homework 3: Speculative Decoding and Quantization

**Total: 50 points.**

| Part | Content | Points |
|---|---|---:|
| 1 | Speculative Sampling (13), Drafting Methods (12), Sampling Correctness (7) | 32 |
| 2 | Group-Wise Quantization (2), AWQ Scaling (2), Scaling Error Analysis (2), Calibration (4), Clipping Search (2), Quantization Tradeoffs (6) | 18 |

Part 1 covers speculative decoding: verification, drafting methods, and sampling correctness. Part 2 covers weight quantization and AWQ: scaling error, calibration, clipping, storage, and throughput. The two parts can be completed independently.

**Environment.** Python 3.10+ and PyTorch 2.2+. All code uses small CPU tensors and requires no pretrained checkpoints.

**Instructions.** Within each part, complete the tasks in order. Replace each `TODO(...)` call with your implementation; a TODO may require several statements. Keep the function signatures and return formats. Cells marked **Provided** need no changes. Run each example after completing the functions it calls.

**Public tests.** Each part ends with a runnable test suite for its coding tasks. Run that part's setup and function-definition cells, then run its public-test cell. You can skip the example cells while testing an incomplete part.

**Submission.** Fill in the written answers, restart the kernel, and run the notebook from top to bottom. Submit the completed `.ipynb` with outputs.

**AI usage.** Using AI is allowed, but you are responsible for understanding your submission.

Please email the TA with questions.


## Part 1: Speculative Decoding (32 points)

### 1.1 Speculative Sampling (13 points)

In each round, a **draft model** proposes candidate tokens and the **target model** scores the proposed sequence. The verifier accepts candidates from left to right, stopping at the first rejection, and appends one replacement or bonus token. This can commit several tokens per round while preserving the target model's output distribution. Treat the drafter as a black box in Section 1.1: its candidates and sampling probabilities are given. [Leviathan et al., 2023](https://proceedings.mlr.press/v202/leviathan23a.html).

#### Systems Perspective

At small batch sizes, autoregressive (AR) decoding is often **memory-bandwidth bound**: each step reads large model weights to produce one token per request. Verification evaluates several candidate positions together and reuses weights across them. This increases **arithmetic intensity** (FLOPs per byte transferred) and can improve **compute utilization**. Accepting several candidates spreads the weight-read cost over multiple output tokens. [SGLang: EAGLE-3 on Vertex](https://www.lmsys.org/blog/2025-12-01-eagle3-vertex/).

Drafting and rejected candidates add work. At high concurrency, ordinary batching already improves utilization, leaving less room for speculation to help. [vLLM: Speculative Decoding](https://docs.vllm.ai/en/latest/features/speculative_decoding/).


#### Setup

Run the provided cell below. `DraftProposal` holds candidate token IDs and the full vocabulary distributions used to sample them; `VerificationResult` holds the verifier's output. The supplied `sample_categorical` draws from a probability row using one uniform random value.

All tensors are on CPU. Sampling uses `torch.float64` probabilities and `torch.long` token IDs. Assume valid shapes, normalized nonnegative probability rows, and independent uniform values in $[0,1)$. Do not modify input tensors. `sample_categorical` returns a scalar `torch.long` tensor.


In [ ]:
# Provided.
from dataclasses import dataclass
from typing import Callable

import torch

# Probability tensors use float64; token IDs use torch.long.
# Assume valid shapes and normalized, nonnegative probability rows.
# Do not mutate input tensors.
torch.set_default_dtype(torch.float64)


def TODO(task):
    raise NotImplementedError(f"Complete {task}")


@dataclass
class DraftProposal:
    tokens: torch.Tensor  # [K], torch.long; K is the draft length
    probs: torch.Tensor   # [K, V], actual proposal probabilities; V is vocabulary size


@dataclass
class VerificationResult:
    committed: torch.Tensor  # accepted prefix + one replacement/bonus token
    accepted_count: int
    next_probs: torch.Tensor  # [V], distribution of the appended token


def sample_categorical(probs, uniform):
    # Provided inverse transform sampling for a normalized [V] row.
    # uniform is in [0, 1); separate draws use independent uniforms.
    # Select the first CDF entry strictly greater than uniform.
    cumulative = probs.cumsum(dim=-1)
    cumulative[-1] = 1.0
    value = torch.as_tensor(uniform, dtype=probs.dtype, device=probs.device)
    return torch.searchsorted(cumulative, value, right=True).long()


#### 1.1.1 Rejection Sampling (8 points)

Let $c$ be the committed prefix, $K$ the number of draft candidates, and $V$ the vocabulary size. Candidate $x_i$ is `proposal.tokens[i]`, for $i=0,\ldots,K-1$.

- `proposal.probs[i]` is the full distribution $q_i$ used to sample $x_i$, given $c$ and the earlier candidates. The drafter may ignore some of this history, but the recorded row must be the one it actually used.
- `target_probs[i]` is the target distribution $p_i$ for the same position, conditioned on $c$ followed by `proposal.tokens[:i]`.
- `target_probs[K]` predicts the token after the full draft. It will be used when all candidates are accepted.

Thus, `proposal.probs` has shape `[K, V]` and `target_probs` has shape `[K+1, V]`. Target rows are supplied here; Section 1.1.2 constructs them.

##### RS-1: Acceptance Probability (1 point)

Accept candidate $x_i$ with probability

$$
a_i=\min\left(1,\frac{p_i(x_i)}{q_i(x_i)}\right).
$$

The acceptance test passes when an independent uniform value is strictly smaller than $a_i$.

Implement `acceptance_probabilities`, returning a tensor of shape `[K]`. Gather $p_i$ and $q_i$ at each candidate ID. Since the candidate was sampled from $q_i$, its denominator is positive. For $K=0$, return an empty vector.


In [ ]:
def acceptance_probabilities(proposal: DraftProposal, target_probs: torch.Tensor):
    # RS-1: [K] candidate acceptance probabilities; empty when K == 0.
    return TODO("RS-1: acceptance probabilities")


##### RS-2: Residual Distribution (1 point)

After a rejection, the replacement token must supply the probability mass missing from the accepted branch. For the current position's full vocabulary distributions $p$ and $q$, use

$$
r(v)=\frac{[p(v)-q(v)]_+}{\sum_u[p(u)-q(u)]_+},
\qquad [z]_+=\max(z,0).
$$

Implement `residual_distribution` for two tensors of shape `[V]`. Return the normalized residual row. The caller uses this function only when the denominator is positive. If $p=q$, rejection cannot occur. Section 1.3 uses probability-mass accounting to explain why this residual recovers the target distribution.


In [ ]:
def residual_distribution(target_probs: torch.Tensor, draft_probs: torch.Tensor):
    # RS-2: [V] residual distribution, called only when residual mass > 0.
    return TODO("RS-2: residual distribution")


##### RS-3: Prefix Verification (6 points)

Implement `verify_proposal` using the two helpers. Verification proceeds from left to right and stops at the first rejection.

For example, if $x_0$ is accepted and $x_1$ is rejected, replacing $x_1$ with $z$ gives the context $(c,x_0,z)$. The target rows for later candidates were computed with $x_1$ in the history, so discard that suffix and draft again from the corrected context.

Let $N$ be the number of consecutive accepted candidates.

- **First rejection at $N<K$.** Keep `proposal.tokens[:N]` and append a sample from the residual of `target_probs[N]` and `proposal.probs[N]`.
- **All $K$ candidates accepted.** Keep the full draft and append a **bonus token** from `target_probs[K]`.
- **Empty proposal, $K=0$.** Sample one token from `target_probs[0]`; the accepted count is zero.

Use `accept_uniforms[i]` for candidate $i$ and `next_uniform` for the replacement or bonus draw. Return a `VerificationResult` containing `N+1` committed tokens, `accepted_count=N`, and the distribution used for the appended token in `next_probs`.

Return the full block here. Section 1.1.2 handles the end-of-sequence (EOS) token and output budget.


In [ ]:
def verify_proposal(
    proposal: DraftProposal,
    target_probs: torch.Tensor,
    accept_uniforms: torch.Tensor,
    next_uniform,
) -> VerificationResult:
    acceptance = acceptance_probabilities(proposal, target_probs)
    accepted_count = TODO("RS-3: accepted-prefix length")
    next_probs = TODO("RS-3: residual or bonus distribution")
    next_token = sample_categorical(next_probs, next_uniform)
    committed = TODO("RS-3: committed tokens")
    return VerificationResult(committed, accepted_count, next_probs)


#### 1.1.2 Speculative Decoding (5 points)

Use the verifier to generate a sequence. First obtain the target rows, then repeat drafting and verification on the updated prefix.

##### SD-1: Target Probability Alignment (1 point)

A target Transformer can evaluate a known draft in one forward pass: causal attention gives each position access only to its preceding context. The distribution used to verify $x_i$ is the prediction **before** $x_i$ is read.

For a three-token proposal:

| Row | Conditioning context | Use |
|---|---|---|
| $p_0$ | $c$ | Verify $x_0$ |
| $p_1$ | $(c,x_0)$ | Verify $x_1$ |
| $p_2$ | $(c,x_0,x_1)$ | Verify $x_2$ |
| $p_3$ | $(c,x_0,x_1,x_2)$ | Sample a bonus token if all candidates are accepted |

The toy interface `target_next_probs(prefix)` returns one `[V]` row at a time. Implement `target_rows_for_proposal` by evaluating the $K+1$ contexts in the table's pattern and stacking their rows. This exercise checks alignment; the individual toy calls stand in for one parallel target pass.

`prefix` is a tuple of Python token IDs. Convert the candidate slice to Python IDs with `tuple(proposal.tokens[:i].tolist())` before appending it to the prefix. For an empty proposal, return the single target row with shape `[1, V]`.


In [ ]:
def target_rows_for_proposal(
    target_next_probs: Callable,
    prefix: tuple,
    proposal: DraftProposal,
) -> torch.Tensor:
    # SD-1: [K+1, V] next-token distributions at the candidate prefixes.
    return TODO("SD-1: target probability alignment")


##### Decoding Loop (4 points)

Implement `generate_speculatively` using `target_rows_for_proposal` and `verify_proposal`. The supplied `propose(prefix, generator)` returns a `DraftProposal`. In this section's opaque drafter, its length depends only on the committed prefix.

| Task | Points | Computation |
|---|---:|---|
| SD-2 | 2 | Propose candidates, obtain target rows, verify, and update the prefix |
| SD-3 | 1 | Apply the output budget and EOS stopping rule |
| SD-4 | 1 | Record each round's trace |

Use the provided generator for all random draws. After each `propose` call, set $K$ to that proposal's length, draw $K$ acceptance uniforms, and then draw one replacement/bonus uniform.

After verification, retain at most the remaining output budget. If `eos_token_id` is set, retain tokens through the first EOS, inclusive, and stop. Append only the retained tokens to `generated` and `current_prefix`. An EOS in a rejected draft suffix has no effect. `max_new_tokens` is nonnegative; zero returns an empty output and trace without calling either model. An empty proposal still emits one target token.

For SD-4, append a dictionary with `prefix_before`, `draft_tokens`, `accepted_count`, `committed_tokens`, and `emitted_tokens`. Store token sequences as tuples of Python integers and `accepted_count` as an integer. `committed_tokens` is the full verifier result; `emitted_tokens` is the portion retained by the stopping rules.

Return the newly generated IDs as a one-dimensional `torch.long` tensor, followed by the trace list. The initial prefix is nonempty and contains no EOS.


In [ ]:
def generate_speculatively(
    target_next_probs: Callable,
    propose: Callable,
    prefix: tuple,
    max_new_tokens: int,
    seed: int = 0,
    eos_token_id: int | None = None,
):
    generator = torch.Generator().manual_seed(seed)
    current_prefix = tuple(prefix)
    generated = []
    trace = []

    # SD-2: draft generation, target verification, rejection sampling.
    # SD-3: stopping criteria. SD-4: iteration trace.
    TODO("SD-2 through SD-4: speculative decoding")

    return torch.tensor(generated, dtype=torch.long), trace


#### Example: Opaque Drafter

The provided `OPAQUE.propose` returns two candidates after A or B and an empty proposal after C. Its vocabulary is A, B, C; treat its probability tables as a black box.

After completing Sections 1.1.1 and 1.1.2, run the examples and inspect which draft tokens are accepted, replaced, or discarded. The first run uses C as EOS. The second starts from C with EOS disabled, so its first round exercises the empty-proposal case.


In [ ]:
# Provided. Do not edit this cell.
A, B, C = 0, 1, 2
VOCAB_SIZE = 3

TARGET_TRANSITION = torch.tensor([
    [0.60, 0.25, 0.15],
    [0.10, 0.65, 0.25],
    [0.35, 0.15, 0.50],
])


def example_target_probs(prefix):
    return TARGET_TRANSITION[prefix[-1]].clone()


class OpaqueDrafter:
    def __init__(self):
        self._probabilities = {
            A: torch.tensor([[0.40, 0.50, 0.10],
                             [0.70, 0.20, 0.10]]),
            B: torch.tensor([[0.30, 0.50, 0.20],
                             [0.10, 0.60, 0.30]]),
        }

    def propose(self, prefix, generator):
        if prefix[-1] == C:
            return DraftProposal(
                torch.empty(0, dtype=torch.long),
                torch.empty(0, VOCAB_SIZE),
            )
        probabilities = self._probabilities[prefix[-1]].clone()
        uniforms = torch.rand(len(probabilities), generator=generator)
        tokens = torch.stack([
            sample_categorical(row, uniform)
            for row, uniform in zip(probabilities, uniforms)
        ])
        return DraftProposal(tokens, probabilities)


OPAQUE = OpaqueDrafter()


In [ ]:
# Run after completing Sections 1.1.1 and 1.1.2.
generated, trace = generate_speculatively(
    example_target_probs, OPAQUE.propose, prefix=(A,),
    max_new_tokens=8, seed=7, eos_token_id=C,
)
print("With EOS:", generated.tolist())
for row in trace:
    print(row)

generated, trace = generate_speculatively(
    example_target_probs, OPAQUE.propose, prefix=(C,),
    max_new_tokens=1, seed=7,
)
print("Empty proposal:", generated.tolist())
print(trace[0])


### 1.2 Drafting Methods (12 points)

The verifier accepts any drafter that returns candidate IDs and the probability rows used to sample them. You will now implement three ways to construct those rows. The model forward passes are supplied; your code handles sampling and token dependencies.

Here $c$ is the committed prefix and $x_{<i}$ denotes the earlier candidates in the current block. The **anchor token**, $x_{-1}$, is the final token of $c$. The examples use small models with the same interfaces as the drafting procedures.


#### 1.2.1 EAGLE-3: Autoregressive Drafting (3 points)

EAGLE-3 generates candidates one at a time. It combines target features from low, middle, and high layers, and predicts tokens through a language-model output head, or **LM head**. [EAGLE-3, Section 3](https://arxiv.org/html/2503.01840v1#S3).

After the first step, each draft state uses the preceding draft state and sampled token, so $q_i$ can depend on $c$ and $x_{<i}$. During training, EAGLE-3 also feeds draft states back into later steps; the paper calls this **training-time test**. Implement the inference recurrence for a single candidate chain here. The model handles feature fusion and the LM head.

The provided model has three methods:

| Interface | Result |
|---|---|
| `model.start(prefix)` | Initial draft state, using context features and the anchor token |
| `model.logits(state)` | Vocabulary logits of shape `[V]` for the next candidate |
| `model.advance(state, token_id)` | Next state after incorporating the sampled token |

At each step, convert logits to probabilities, sample a token, and pass that token into the next state:

$$
q_i=\operatorname{softmax}(\operatorname{LMHead}(s_i)/T),
\qquad x_i\sim q_i,
\qquad s_{i+1}=F(s_i,\operatorname{Embed}(x_i)).
$$

The supplied `logits_to_probs` applies softmax over the vocabulary at temperature $T>0$. These temperature-scaled rows are the proposal probabilities used by the verifier.

**Task.** Implement `draft_eagle3` for `num_tokens >= 1`.

| Task | Points | Computation |
|---|---:|---|
| AR-1 | 2 | Initialize the state, sample candidates, and feed sampled tokens back into the model |
| AR-2 | 1 | Return candidate IDs and their full conditional probability rows |

Use `uniforms[i]` for candidate $i$. Keep scalar token tensors for the returned `[K]` tensor; pass a Python integer to `model.advance`. Advance only when another candidate remains. Return a `DraftProposal` with probability rows of shape `[K, V]`.


In [ ]:
# Provided.
def logits_to_probs(logits, temperature=1.0):
    # Sampling exercises use strictly positive temperature.
    return torch.softmax(logits / temperature, dim=-1)


In [ ]:
def draft_eagle3(
    model,
    prefix: tuple,
    num_tokens: int,
    uniforms: torch.Tensor,
    temperature: float = 1.0,
) -> DraftProposal:
    sampled_tokens = []
    probability_rows = []

    # AR-1: recurrent state and autoregressive sampling.
    # AR-2: conditional proposal probabilities.
    TODO("AR-1 and AR-2: autoregressive drafting")

    return DraftProposal(
        torch.stack(sampled_tokens),
        torch.stack(probability_rows),
    )


#### 1.2.2 DFlash: Parallel Drafting (2 points)

DFlash uses a **block diffusion drafter**: a masked block is predicted in one forward pass, conditioned on target features at each draft layer. This produces all $K$ vocabulary-logit rows before sampling the block. [DFlash, Section 4.1](https://arxiv.org/html/2602.06036v1#S4.SS1).

Hidden states can interact through block attention. Once the context and logits are fixed, however, independent categorical draws give

$$
Q(x_0,\ldots,x_{K-1}\mid c)=\prod_{i=0}^{K-1}q_i(x_i\mid c).
$$

An earlier draw does not change the later rows. For example, suppose a two-token target assigns probability $1/2$ each to AA and BB. Both token marginals are uniform, but sampling them independently also produces AB and BA. Rejection sampling still corrects the output distribution; the proposal simply misses this dependence.

**Task.** Implement `draft_dflash` for `num_tokens >= 1`. The supplied `block_model(prefix, num_tokens)` returns logits of shape `[K, V]`.

| Task | Points | Computation |
|---|---:|---|
| PD-1 | 1 | Call the block model once and apply vocabulary softmax with temperature |
| PD-2 | 1 | Sample independently from each row and assemble the proposal |

Use `uniforms[i]` for position $i$. Return candidate IDs `[K]` and the sampling rows `[K, V]`. Keep later probability rows fixed after sampling earlier tokens.


In [ ]:
def draft_dflash(
    block_model: Callable,
    prefix: tuple,
    num_tokens: int,
    uniforms: torch.Tensor,
    temperature: float = 1.0,
) -> DraftProposal:
    # PD-1: one backbone forward pass, then vocabulary softmax.
    probability_rows = TODO("PD-1: parallel proposal distributions")
    sampled_tokens = TODO("PD-2: independent categorical sampling")
    return DraftProposal(sampled_tokens, probability_rows)


#### 1.2.3 DSpark: Semi-Autoregressive Drafting (4 points)

DFlash's probability rows stay fixed as tokens are sampled. DSpark adds a lightweight sequential head that adjusts each row using earlier draws. Its parallel backbone still computes the block in one pass. This exercise uses DeepSpec's **vanilla Markov head**, which conditions each row on the immediately preceding token. [DSpark, Section 3.1](https://arxiv.org/html/2607.05147v1#S3.SS1).

Given the previous token $u$, the head adds a low-rank logit bias:

$$
b_i=W_1[u]W_2,\qquad
\ell_i=U_i+b_i,\qquad
q_i=\operatorname{softmax}(\ell_i/T).
$$

| Symbol | Function parameter | Shape |
|---|---|---|
| $U$ | `base_logits` | `[K, V]` |
| $W_1$ | `transition_embedding` | `[V, r]` |
| $W_2$ | `transition_projection` | `[r, V]` |

Here $r$ is the rank of the correction. At position zero, $u$ is the anchor token; afterward, it is the previous sampled candidate. The resulting proposal is $q_i(x_i\mid c,x_{i-1})$. Its base logits still use the full committed context $c$.

The correction follows [DeepSpec's `VanillaMarkov`](https://github.com/deepseek-ai/DeepSpec/blob/005e03b81cec38b7da6399833d609ee89a2587f2/deepspec/modeling/dspark/markov_head.py). The matrices parameterize logits and may contain negative values.

**Task.** Implement `draft_markov` for $K\geq1$.

| Task | Points | Computation |
|---|---:|---|
| MK-1 | 2 | Compute the previous-token bias, corrected logits, and temperature-scaled probabilities |
| MK-2 | 2 | Sample, save the token and its probability row, and update the previous token |

Compute the current `[V]` row from `transition_embedding[previous_token]` and `transition_projection`, then use `uniforms[position]` for sampling. Return candidate IDs `[K]` and the corrected probability rows `[K, V]`. The verifier needs these rows because they generated the candidates.


In [ ]:
def draft_markov(
    base_logits: torch.Tensor,            # [K, V]
    transition_embedding: torch.Tensor,   # [V, r]
    transition_projection: torch.Tensor,  # [r, V]
    anchor_token: int,
    uniforms: torch.Tensor,               # [K]
    temperature: float = 1.0,
) -> DraftProposal:
    previous_token = int(anchor_token)
    sampled_tokens = []
    probability_rows = []

    for position in range(base_logits.shape[0]):
        # MK-1: compute the current row using the previous sampled token.
        corrected_logits = TODO("MK-1: Markov logit correction")
        probabilities = TODO("MK-1: proposal distribution")

        # MK-2: retain the distribution used for this draw.
        token = TODO("MK-2: categorical sampling")
        TODO("MK-2: record token and q, then update previous_token")

    return DraftProposal(
        torch.stack(sampled_tokens),
        torch.stack(probability_rows),
    )


#### 1.2.4 Confidence-Based Prefix Selection (3 points)

A long draft may contain an unlikely suffix. DSpark's **confidence head** estimates acceptance before target verification, allowing the scheduler to choose how much of the block to verify.

At a fixed draft history, the expected acceptance probability for position $i$ is

$$
\beta_i=\mathbb{E}_{X_i\sim q_i}[a_i(X_i)]
=\sum_v\min(p_i(v),q_i(v)).
$$

This averages RS-1's test over the candidate at position $i$, for a fixed history. The confidence head estimates this probability from the block features and previous-token embedding; a sigmoid maps its logit to a score. The estimate concerns this position's acceptance, while committing it also requires every earlier candidate to pass. Computing $\beta_i$ exactly would require target verification. [DSpark, Section 3.2.1](https://arxiv.org/html/2607.05147v1#S3.SS2.SSS1).

Use the fixed-threshold prefix rule in [DeepSpec's draft adapter](https://github.com/deepseek-ai/DeepSpec/blob/005e03b81cec38b7da6399833d609ee89a2587f2/deepspec/eval/dspark/draft_ops.py). This is a simplified alternative to the paper's hardware-aware scheduler: keep positions before the first score below the threshold.

**Non-anticipation.** The decision to include candidate $i$ may depend on $c$ and $x_{<i}$, but cannot use $x_i$ or later candidates. Even if the whole block has already been sampled, each inclusion decision must obey this information restriction. Section 1.3.2 examines what happens when it does not.

**Task.** Implement `select_confident_prefix`. The provided `confidence_head(previous_tokens)` returns logits of shape `[K]`; row $i$ uses only its fixed block features and `previous_tokens[i]`.

| Task | Points | Computation |
|---|---:|---|
| CF-1 | 1 | Build `previous_tokens`: the anchor followed by all candidates except the last |
| CF-2 | 1 | Apply sigmoid and find the first score below the threshold |
| CF-3 | 1 | Return matching token and probability prefixes |

Return an empty input unchanged, without calling the confidence head. Otherwise, keep rows before the first score strictly below `threshold`. A score equal to the threshold passes; if every score passes, keep the whole block. A failure at position zero returns an empty proposal. The threshold is in $[0,1]$. Retain the original probability rows without renormalizing them: selection changes the block length, while the verifier still needs each draw's actual $q_i$.


In [ ]:
def select_confident_prefix(
    proposal: DraftProposal,
    confidence_head: Callable,
    anchor_token: int,
    threshold: float,
) -> DraftProposal:
    # CF-1: previous-token alignment.
    # CF-2: confidence threshold. CF-3: prefix truncation.
    return TODO("CF-1 through CF-3: confidence-based prefix selection")


#### Example: Drafter Integration

The provided toy models expose the interfaces used in Sections 1.2.1–1.2.4. Each wrapper returns a `DraftProposal` to `generate_speculatively` from Section 1.1.2.

| Drafter | Distribution at position $i$ | Computation |
|---|---|---|
| EAGLE-3 recurrence | $q_i(x_i\mid c,x_{<i})$ | Update the draft state after each sampled token |
| DFlash | $q_i(x_i\mid c)$ | Sample from fixed parallel rows |
| DSpark Markov head | $q_i(x_i\mid c,x_{i-1})$ | Correct each parallel row using the previous token |

The DFlash and DSpark examples use identical base logits. Inspect their recorded probability rows to see the Markov correction; the confidence-selected variant also changes the proposal length. All four wrappers use the same verifier.


In [ ]:
# Provided. Do not edit this cell.
class ExampleEagleModel:
    def __init__(self):
        self.transition = torch.tensor([
            [0.70, 0.20, 0.10],
            [0.20, 0.60, 0.20],
            [0.20, 0.30, 0.50],
        ])

    def start(self, prefix):
        return tuple(prefix)

    def logits(self, state):
        return self.transition[state[-1]].log()

    def advance(self, state, token_id):
        return state + (int(token_id),)


def example_block_logits(prefix, num_tokens):
    position_bias = torch.tensor([
        [0.20, -0.10, 0.00],
        [-0.10, 0.20, 0.00],
        [0.00, -0.10, 0.20],
    ])
    positions = torch.arange(num_tokens) % len(position_bias)
    return TARGET_TRANSITION[prefix[-1]].log()[None, :] + position_bias[positions]


MARKOV_EMBEDDING = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [-1.0, -1.0],
])
MARKOV_PROJECTION = torch.tensor([
    [0.40, -0.20, -0.20],
    [-0.20, 0.40, -0.20],
])


def example_confidence_head(previous_tokens):
    # Each row uses only its own previous token and fixed position features.
    score_table = torch.tensor([
        [0.90, 0.85, 0.70],
        [0.84, 0.75, 0.65],
        [0.78, 0.70, 0.60],
    ])
    positions = torch.arange(previous_tokens.numel()) % len(score_table)
    return torch.logit(score_table[positions, previous_tokens])


def make_example_proposers(num_tokens=2, temperature=1.0):
    eagle_model = ExampleEagleModel()

    def eagle(prefix, generator):
        uniforms = torch.rand(num_tokens, generator=generator)
        return draft_eagle3(
            eagle_model, prefix, num_tokens, uniforms, temperature,
        )

    def dflash(prefix, generator):
        uniforms = torch.rand(num_tokens, generator=generator)
        return draft_dflash(
            example_block_logits, prefix, num_tokens, uniforms, temperature,
        )

    def dspark(prefix, generator):
        base_logits = example_block_logits(prefix, num_tokens)
        uniforms = torch.rand(num_tokens, generator=generator)
        return draft_markov(
            base_logits, MARKOV_EMBEDDING, MARKOV_PROJECTION,
            anchor_token=prefix[-1], uniforms=uniforms, temperature=temperature,
        )

    def dspark_confident(prefix, generator):
        proposal = dspark(prefix, generator)
        return select_confident_prefix(
            proposal, example_confidence_head, prefix[-1], threshold=0.80,
        )

    return {
        "EAGLE-3-style": eagle,
        "DFlash-style": dflash,
        "DSpark Markov": dspark,
        "DSpark with confidence": dspark_confident,
    }


In [ ]:
# Run after completing Sections 1.1 and 1.2.
for name, propose in make_example_proposers().items():
    generated, trace = generate_speculatively(
        example_target_probs, propose, prefix=(A,), max_new_tokens=6, seed=7,
    )
    print(name, generated.tolist())
    for row in trace:
        print(" ", row)


### 1.3 Sampling Correctness (7 points)

The verifier in Section 1.1 uses the proposal distribution for both acceptance and residual sampling. Section 1.2 added a Markov correction and a prefix selector. The questions below examine errors in recording that distribution and in selecting a prefix.

First, fix one history and account for the accepted and replacement probability mass. [Leviathan et al., Appendix A.1](https://proceedings.mlr.press/v202/leviathan23a/leviathan23a.pdf).

For $q(v)>0$, token $v$ is proposed with probability $q(v)$ and accepted with probability $\min(1,p(v)/q(v))$. Its accepted mass is $\min(p(v),q(v))$. A token with $q(v)=0$ contributes no accepted mass.

The total rejection probability is

$$
Z=1-\sum_v\min(p(v),q(v))
=\sum_v[p(v)-q(v)]_+.
$$

When $Z>0$, sample the replacement from $r(v)=[p(v)-q(v)]_+/Z$. The total output mass is

$$
\Pr(Y=v)
=\underbrace{\min(p(v),q(v))}_{\text{accepted mass}}
+\underbrace{Zr(v)}_{\text{replacement mass}}
=\min(p(v),q(v))+[p(v)-q(v)]_+
=p(v).
$$

If $p=q$, rejection has zero probability and the accepted branch already has the target distribution. Applying this rule at each committed history preserves the sequence distribution. Runs with the same seed need not produce identical sequences.

Use this argument in the questions below. Show the accepted and replacement contributions, then report the resulting distribution.


#### 1.3.1 Proposal Distribution Mismatch (4 points)

Consider a one-token DSpark proposal over `[A, B]`. The base softmax is $q_{\mathrm{base}}=(0.5,0.5)$; after the Markov correction, the sampler uses $q_{\mathrm{actual}}=(0.9,0.1)$. The target is $p=(0.6,0.4)$.

A buggy implementation samples from $q_{\mathrm{actual}}$ but stores $q_{\mathrm{base}}$ in `DraftProposal.probs`. Its verifier uses the recorded row for both acceptance and residual sampling. Consider only the **first committed token**, even if the verifier also emits a bonus token.

1. **(1 point)** Derive the first output token's distribution. Show the accepted and replacement contributions.
2. **(2 points)** A patch changes only the acceptance denominator to $q_{\mathrm{actual}}$ and still uses $q_{\mathrm{base}}$ in the residual. Derive the output distribution and determine whether this fixes the bug.
3. **(1 point)** Identify the complete fix and show how the accepted and replacement masses recover $p$.


**Answer**

Write your answer here.


#### 1.3.2 Selection Bias (3 points)

A one-token drafter samples $X\sim q=(0.5,0.5)$ over `[A, B]`, while $p=(0.7,0.3)$.

A selector inspects $X$. If $X=A$, it keeps the candidate and runs the standard verifier with the original $q$. If $X=B$, it discards the proposal and draws a fresh token directly from $p$. Consider only the first committed token.

1. **(2 points)** Derive the output distribution and identify the information used by the selector that causes bias.
2. **(1 point)** State a valid restriction on the decision to include position $i$. Explain why Section 1.2.4's previous-token inputs and first-failure rule meet it.

This example is adapted from [DSpark, Appendix A](https://arxiv.org/html/2607.05147v1#A1).


**Answer**

Write your answer here.


### Public Tests — Part 1

Run this cell after running the setup and function definitions for Part 1. Each check reports `ok`, `FAIL`, or `ERROR`; unfinished TODOs report an error. `OK` at the end means all public cases in this part passed.


In [ ]:
# Public tests for Part 1. Run after executing the Part 1 function definitions.
import sys
import unittest


class SpeculativeDecodingPublicTests(unittest.TestCase):
    def test_rs1_candidate_ratios(self):
        proposal = DraftProposal(
            torch.tensor([1, 0]), torch.tensor([[0.5, 0.5], [0.5, 0.5]]),
        )
        rows = torch.tensor([[0.875, 0.125], [0.75, 0.25], [0.5, 0.5]])
        torch.testing.assert_close(
            acceptance_probabilities(proposal, rows), torch.tensor([0.25, 1.0]),
        )

    def test_rs2_normalized_positive_residual(self):
        target = torch.tensor([0.25, 0.25, 0.50])
        draft = torch.tensor([0.625, 0.125, 0.25])
        torch.testing.assert_close(
            residual_distribution(target, draft), torch.tensor([0.0, 1 / 3, 2 / 3]),
        )
        torch.testing.assert_close(target, torch.tensor([0.25, 0.25, 0.50]))
        torch.testing.assert_close(draft, torch.tensor([0.625, 0.125, 0.25]))

    def test_rs3_rejection_at_the_threshold_discards_suffix(self):
        proposal = DraftProposal(torch.tensor([0, 1, 2]), torch.eye(3))
        rows = torch.tensor([[1.0, 0.0, 0.0], [0.25, 0.50, 0.25],
                             [0.0, 0.0, 1.0], [0.0, 1.0, 0.0]])
        result = verify_proposal(
            proposal, rows, torch.tensor([0.0, 0.5, 0.0]), next_uniform=0.625,
        )
        self.assertEqual(result.accepted_count, 1)
        torch.testing.assert_close(result.committed, torch.tensor([0, 2]))
        torch.testing.assert_close(result.next_probs, torch.tensor([0.5, 0.0, 0.5]))

    def test_sd1_target_row_alignment(self):
        contexts = {
            (2, 0): [0.1, 0.2, 0.7],
            (2, 0, 1): [0.3, 0.5, 0.2],
            (2, 0, 1, 2): [0.6, 0.3, 0.1],
        }

        def target(prefix):
            self.assertTrue(all(type(token) is int for token in prefix))
            return torch.tensor(contexts[prefix])

        proposal = DraftProposal(torch.tensor([1, 2]), torch.eye(3)[1:])
        torch.testing.assert_close(
            target_rows_for_proposal(target, (2, 0), proposal),
            torch.tensor([[0.1, 0.2, 0.7], [0.3, 0.5, 0.2], [0.6, 0.3, 0.1]]),
        )

    def test_sd2_sd3_sd4_prefix_stopping_and_trace(self):
        target_tokens = {
            (0,): 0, (0, 1): 2, (0, 1, 2): 1,
            (0, 0): 1, (0, 0, 1): 2, (0, 0, 1, 2): 0,
        }
        vocabulary = torch.eye(3)
        draws = []

        def target(prefix):
            return vocabulary[target_tokens[prefix]].clone()

        def propose(prefix, generator):
            draws.append(torch.rand((), generator=generator))
            return DraftProposal(torch.tensor([1, 2]), vocabulary[1:].clone())

        generated, trace = generate_speculatively(
            target, propose, (0,), max_new_tokens=4, seed=7, eos_token_id=2,
        )
        torch.testing.assert_close(generated, torch.tensor([0, 1, 2]))
        self.assertEqual(trace, [
            dict(prefix_before=(0,), draft_tokens=(1, 2), accepted_count=0,
                 committed_tokens=(0,), emitted_tokens=(0,)),
            dict(prefix_before=(0, 0), draft_tokens=(1, 2), accepted_count=2,
                 committed_tokens=(1, 2, 0), emitted_tokens=(1, 2)),
        ])
        expected_draws = torch.rand(5, generator=torch.Generator().manual_seed(7))
        torch.testing.assert_close(torch.stack(draws), expected_draws[[0, 4]])

        generated, trace = generate_speculatively(
            target, propose, (0,), max_new_tokens=2, eos_token_id=2,
        )
        torch.testing.assert_close(generated, torch.tensor([0, 1]))
        self.assertEqual(trace[-1]["emitted_tokens"], (1,))

    def test_ar1_ar2_temperature_and_sampled_token_feedback(self):
        logits_by_state = {
            (0, 1): [1.0, 9.0],
            (0, 1, 1): [16.0, 1.0],
            (0, 1, 1, 0): [9.0, 4.0],
        }

        class ToyModel:
            def start(self, prefix):
                return prefix

            def logits(self, state):
                return torch.tensor(logits_by_state[state]).log()

            def advance(self, state, token_id):
                if type(token_id) is not int or state == (0, 1, 1, 0):
                    raise AssertionError("Advance with a Python int only when another token remains.")
                return state + (token_id,)

        result = draft_eagle3(
            ToyModel(), (0, 1), 3, torch.tensor([0.30, 0.65, 0.70]), temperature=2.0,
        )
        torch.testing.assert_close(result.tokens, torch.tensor([1, 0, 1]))
        torch.testing.assert_close(
            result.probs, torch.tensor([[0.25, 0.75], [0.8, 0.2], [0.6, 0.4]]),
        )

    def test_pd1_pd2_one_block_and_independent_draws(self):
        calls = []

        def block_model(prefix, num_tokens):
            calls.append((prefix, num_tokens))
            return torch.tensor([[1.0, 9.0, 4.0], [9.0, 1.0, 4.0]]).log()

        result = draft_dflash(
            block_model, (0, 2), 2, torch.tensor([0.80, 0.20]), temperature=2.0,
        )
        self.assertEqual(calls, [((0, 2), 2)])
        torch.testing.assert_close(result.tokens, torch.tensor([2, 0]))
        torch.testing.assert_close(
            result.probs, torch.tensor([[1 / 6, 1 / 2, 1 / 3], [1 / 2, 1 / 6, 1 / 3]]),
        )

    def test_mk1_mk2_condition_on_anchor_then_previous_sample(self):
        log_two = torch.tensor(2.0).log()
        result = draft_markov(
            base_logits=log_two * torch.tensor([[0.0, 2.0], [0.0, 0.0], [2.0, 0.0]]),
            transition_embedding=torch.eye(2),
            transition_projection=log_two * torch.tensor([[2.0, 0.0], [-2.0, 2.0]]),
            anchor_token=1, uniforms=torch.tensor([0.08, 0.80, 0.40]), temperature=2.0,
        )
        torch.testing.assert_close(result.tokens, torch.tensor([0, 1, 1]))
        torch.testing.assert_close(
            result.probs, torch.tensor([[1 / 9, 8 / 9], [2 / 3, 1 / 3], [1 / 3, 2 / 3]]),
        )

    def test_cf1_cf2_cf3_alignment_threshold_and_matching_rows(self):
        proposal = DraftProposal(torch.tensor([2, 0, 1]), torch.eye(3)[[2, 0, 1]])

        def head(previous_tokens):
            torch.testing.assert_close(previous_tokens, torch.tensor([1, 2, 0]))
            return torch.tensor([0.0, -2.0, 2.0])

        result = select_confident_prefix(proposal, head, 1, threshold=0.5)
        torch.testing.assert_close(result.tokens, torch.tensor([2]))
        torch.testing.assert_close(result.probs, torch.tensor([[0.0, 0.0, 1.0]]))


unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(SpeculativeDecodingPublicTests)
)


## Part 2: Weight Quantization and AWQ (18 points)

Weight quantization stores model parameters with fewer bits, reducing storage and memory traffic. Rounding also changes layer outputs. This part studies that tradeoff using a linear layer and **Activation-aware Weight Quantization (AWQ)**.

Start with group-wise quantization, then examine how channel scaling changes the error. Use calibration inputs to choose channel scales and clipping thresholds. AWQ is post-training quantization: the weights are fixed, and calibration uses no optimizer or backpropagation. [AWQ project](https://hanlab.mit.edu/projects/awq).

The code uses small CPU tensors to simulate quantization and dequantization. It measures numerical error, with no packed kernels or timing experiments. Section 2.6 uses a separate analytical model to examine storage and decode throughput.


### 2.1 Group-Wise Affine Quantization (2 points)

Let $W$ have shape $[O,H]$, where $O$ is the output width and $H$ is the input width. Split each output row into consecutive groups of $G$ weights. A group shares a quantization step $\Delta$ and an integer zero point $z$. For $b$ bits, define

$$
q_{\max}=2^b-1,\qquad
\Delta=\frac{\max(w)-\min(w)}{q_{\max}},\qquad
z=\operatorname{clamp}\!\left(-\operatorname{round}(\min(w)/\Delta),0,q_{\max}\right).
$$

Encode and reconstruct each weight with

$$
q=\operatorname{clamp}\!\left(\operatorname{round}(w/\Delta)+z,0,q_{\max}\right),
\qquad
\widehat w=\Delta(q-z).
$$

Use the [official AWQ quantizer's convention](https://github.com/mit-han-lab/llm-awq/blob/main/awq/quantize/quantizer.py): clamp the range $\max(w)-\min(w)$ to at least $10^{-5}$ **before** dividing by $q_{\max}$. Use `torch.round`.

- **Q-1 (1 point):** Implement `quantize_weight_groups`. Return codes with shape `[O, H]` and scales/zero points with shape `[O, H // group_size]`.
- **Q-2 (1 point):** Implement `dequantize_weight_groups`, returning floating-point weights of shape `[O, H]`.

Codes and zero points use `torch.long`; scales retain the weight dtype. The integer tensors simulate codes and are not bit-packed. Assume finite inputs, `bits >= 2`, and `H` divisible by a positive `group_size`. Do not modify inputs. Applying this quantizer directly to $W$ is the **round-to-nearest (RTN)** baseline.


In [ ]:
# Provided. This setup is independent of Part 1.
from dataclasses import dataclass

import torch

torch.set_default_dtype(torch.float64)


def TODO(task):
    raise NotImplementedError(f"Complete {task}")


@dataclass
class GroupQuantizedWeight:
    codes: torch.Tensor        # [O, H], torch.long
    scales: torch.Tensor       # [O, H // G], floating point
    zero_points: torch.Tensor  # [O, H // G], torch.long


In [ ]:
def quantize_weight_groups(
    weight: torch.Tensor,
    bits: int = 4,
    group_size: int = 4,
) -> GroupQuantizedWeight:
    return TODO("Q-1: group ranges, scales, zero points, and integer codes")


def dequantize_weight_groups(
    quantized: GroupQuantizedWeight,
    group_size: int = 4,
) -> torch.Tensor:
    return TODO("Q-2: reconstruct each weight using its group's scale and zero point")


### 2.2 Activation-Aware Scaling (2 points)

A weight error in input channel $j$ contributes in proportion to that channel's activation. AWQ uses this information to redistribute quantization error while keeping every weight at the chosen bit width.

For row-wise inputs $X\in\mathbb{R}^{N\times H}$ and positive input-channel scales $s\in\mathbb{R}^H$,

$$
XW^\top=(X/s)(W\odot s)^\top.
$$

Division and multiplication broadcast over the **input** dimension. The equality holds before quantization; afterward, use

$$
\widehat Y=(X/s)\,Q(W\odot s)^\top,
$$

where $Q$ includes quantization and dequantization. Scaling changes how channels share each group's quantization range.

**AWQ-1 (1 point).** Implement `activation_channel_scales` using calibration inputs `[N, H]`. Compute $a_j=\operatorname{mean}_n|X_{nj}|$, then

$$
u_j=\max(a_j^\alpha,10^{-4}),\qquad
s_j=\frac{u_j}{\sqrt{\max(u)\min(u)}}.
$$

This follows the [official scale search](https://github.com/mit-han-lab/llm-awq/blob/main/awq/quantize/auto_scale.py). Here $\alpha\in[0,1]$; $\alpha=0$ gives all-one scales.

**AWQ-2 (1 point).** Implement `awq_linear`. Its `scaled_weight` argument is already $Q(W\odot s)$. Return `[N, O]` outputs, applying the inverse activation scaling. Activations remain floating point.


In [ ]:
def activation_channel_scales(
    calibration_inputs: torch.Tensor,  # [N, H]
    alpha: float,
) -> torch.Tensor:                     # [H]
    return TODO("AWQ-1: activation statistics and normalized channel scales")


def awq_linear(
    inputs: torch.Tensor,              # [N, H]
    scaled_weight: torch.Tensor,       # [O, H], quantized then dequantized W*s
    channel_scales: torch.Tensor,      # [H]
) -> torch.Tensor:
    return TODO("AWQ-2: linear output with inverse activation scaling")


### 2.3 Scaling Error Analysis (2 points)

Use two-bit quantization with one group of four weights. Consider the linear layer

$$
W=\begin{bmatrix}-1 & 0.4 & 0.6 & 2\end{bmatrix},
\qquad
X=\operatorname{diag}(1,10,2,2).
$$

The second input channel has the largest activation magnitude. Compare the three scale vectors below, using them directly without normalization. For each, quantize $W\odot s$ with Section 2.1's affine quantizer and define the effective weight $\widehat W=Q(W\odot s)/s$.

**Calculation (1 point).** Complete the table. Output MSE averages the four elements of $X\widehat W^\top-XW^\top$. You may use your helpers, but show the quantization step and effective weights.

| Channel scales $s$ | Group step $\Delta$ | Effective weight $\widehat W$ | Output MSE |
|---|---:|---|---:|
| $(1,1,1,1)$ | | | |
| $(1,2,1,1)$ | | | |
| $(1,8,1,1)$ | | | |

**Explanation (1 point).** Why does moderate scaling improve the result, while increasing the second channel's scale from 2 to 8 makes the overall MSE worse? Compare that channel's error with the errors in the other channels.


**Answer**

Write your answer here.


### 2.4 Calibration (4 points)

The previous example shows that protecting one channel can hurt others. Calibration selects scales by measuring their combined effect on the layer output.

#### Calibration Objective (1 point)

Let $\widehat W=Q(W\odot s)/s$ be the effective weight after scaling, quantization, and inverse scaling. Write the output error $\widehat Y-Y$ in terms of $X$, $W$, and $\widehat W$. Use it to explain why AWQ selects scales by calibration **output MSE** rather than **weight MSE**.


**Answer**

Write your answer here.


#### Scale Search (3 points)

Search the candidate scales using the output MSE from the preceding question.

Implement `search_awq_scales` over the supplied `alphas`, in order:

1. **(1 point)** For each $\alpha$, compute $s$, scale the original weight columns, and quantize/dequantize $W\odot s$. Start every candidate from the original $W$.
2. **(1 point)** Use `awq_linear` to compute $\widehat Y$, then record the mean squared error over all elements of $\widehat Y-X_{\mathrm{cal}}W^\top$. Do not select using weight MSE.
3. **(1 point)** Return the candidate with the smallest calibration error. Break ties by keeping the first candidate. Include every `(alpha, mse)` pair in `calibration_errors`, as Python floats.

The nonempty candidate list includes 0, so the search includes RTN. Keep the original weight and calibration inputs unchanged. Held-out inputs are used only for evaluation.

Select channel scales without clipping in this search. Section 2.5 then searches clipping thresholds with these scales fixed. [AWQ, Section 3.2](https://arxiv.org/html/2306.00978v5#S3.SS2).


In [ ]:
# Provided.
@dataclass
class AWQResult:
    quantized_weight: GroupQuantizedWeight  # codes for the chosen W*s
    channel_scales: torch.Tensor            # [H]
    alpha: float
    calibration_errors: list[tuple[float, float]]


In [ ]:
@torch.no_grad()
def search_awq_scales(
    weight: torch.Tensor,
    calibration_inputs: torch.Tensor,
    bits: int = 4,
    group_size: int = 4,
    alphas: tuple = (0.0, 0.25, 0.5, 0.75, 1.0),
) -> AWQResult:
    return TODO("AWQ-3: quantize candidates, measure output MSE, and select scales")


### 2.5 Clipping Search (2 points)

Scaling redistributes quantization error across input channels. **Clipping** offers another tradeoff: reducing a group's weight range can make its quantization step smaller, at the cost of changing weights outside the retained range.

Keep the channel scales chosen in Section 2.4 fixed. Let $U=W\odot s$. For each output row and weight group $g$, compute $m_g=\max_{j\in g}|U_j|$. A candidate shrink ratio $\rho$ gives

$$
C_{\rho,j}=\operatorname{clamp}(U_j,-\rho m_g,\rho m_g).
$$

Quantize/dequantize $C_\rho$ using the same group-wise quantizer, then evaluate

$$
\widehat Y_\rho=(X_{\mathrm{cal}}/s)\,Q(C_\rho)^\top.
$$

Compare against the **original, unclipped** output $X_{\mathrm{cal}}W^\top$. Comparing against the clipped floating-point output would omit clipping's own error.

Complete `search_awq_clipping`. The loop and best-candidate bookkeeping are provided.

- **CLIP-1 (1 point):** Compute each group's bound, clip the original scaled weights, and quantize the clipped weights.
- **CLIP-2 (1 point):** Apply inverse activation scaling and calculate the output MSE against the original reference.

The supplied ratios lie in $(0,1]$ and include 1, which applies no clipping. Try them in order; the provided strict comparison retains the first minimum. Keep inputs unchanged and use only calibration inputs during the search. Return the quantized weights for the chosen candidate, its ratio, and the full error history.

This exercise uses one shrink ratio for the whole layer and scores the full layer output. The [official AWQ clipping search](https://github.com/mit-han-lab/llm-awq/blob/main/awq/quantize/auto_clip.py) selects thresholds separately for each output-row/group. Both use calibration error to balance clipping and rounding.


In [ ]:
# Provided.
@dataclass
class ClippingResult:
    quantized_weight: GroupQuantizedWeight  # codes for clipped W*s
    ratio: float
    calibration_errors: list[tuple[float, float]]


In [ ]:
@torch.no_grad()
def search_awq_clipping(
    weight: torch.Tensor,              # [O, H], original unquantized weights
    calibration_inputs: torch.Tensor,  # [N, H]
    channel_scales: torch.Tensor,      # [H], fixed result from the scale search
    bits: int = 4,
    group_size: int = 4,
    ratios: tuple = (1.0, 0.98, 0.96, 0.94, 0.92),
) -> ClippingResult:
    reference_outputs = calibration_inputs @ weight.T
    scaled_groups = (weight * channel_scales).reshape(
        weight.shape[0], -1, group_size,
    )
    group_max = scaled_groups.abs().amax(dim=-1, keepdim=True)
    best_error = float("inf")
    best_weight = None
    best_ratio = None
    calibration_errors = []

    for ratio in ratios:
        bound = TODO("CLIP-1: per-group clipping bound")
        clipped_groups = TODO("CLIP-1: clip the original scaled weights")
        quantized_weight = TODO("CLIP-1: quantize the clipped weights")
        dequantized_weight = dequantize_weight_groups(
            quantized_weight, group_size,
        )
        outputs = TODO("CLIP-2: output with inverse activation scaling")
        error = TODO("CLIP-2: MSE against the original unclipped output")

        calibration_errors.append((float(ratio), float(error)))
        if error < best_error:
            best_error = float(error)
            best_weight = quantized_weight
            best_ratio = float(ratio)

    return ClippingResult(best_weight, best_ratio, calibration_errors)


#### Example: Scaling, Clipping, and Held-Out Error

The linear layer below has three outputs and eight input channels. Some channels have large activations; others have large weights but small activations. Use eight diagonal calibration rows to select channel scales, then a clipping ratio. Evaluate the resulting weights on four held-out mixtures of the input channels.

Compare RTN, scaling alone, and scaling followed by clipping. Including $\alpha=0$ and $\rho=1$ ensures that each search cannot worsen its calibration objective relative to the previous stage. That guarantee does not extend to the held-out inputs. No timing measurement is required.


In [ ]:
# Provided. Run after completing Q-1 through CLIP-2.
projection_weight = torch.tensor([
    [-0.23, -1.00,  0.80,  4.00,  0.17, -0.70,  0.90, -3.00],
    [ 0.31,  0.90, -0.80, -3.50, -0.21,  0.60, -0.70,  4.00],
    [-0.15, -0.60,  0.70,  3.00,  0.26, -0.90,  1.00, -4.00],
])
feature_magnitudes = torch.tensor([8.0, 1.0, 1.0, 0.1, 4.0, 1.0, 1.0, 0.1])
calibration_inputs = torch.diag(feature_magnitudes)
heldout_inputs = torch.tensor([
    [ 1.0,  1.0, -1.0,  1.0,  0.5, -0.5,  1.0, -1.0],
    [-1.0,  0.5,  1.0, -1.0,  1.0,  1.0, -0.5,  0.5],
    [ 0.5, -1.0,  0.5,  1.0, -1.0,  0.5,  1.0,  1.0],
    [ 1.0, -0.5, -1.0,  0.5,  0.5,  1.0, -1.0, -0.5],
]) * feature_magnitudes

scale_result = search_awq_scales(projection_weight, calibration_inputs)
clip_result = search_awq_clipping(
    projection_weight, calibration_inputs, scale_result.channel_scales,
)
rtn_weight = dequantize_weight_groups(
    quantize_weight_groups(projection_weight),
)
scaled_weight = dequantize_weight_groups(scale_result.quantized_weight)
clipped_weight = dequantize_weight_groups(clip_result.quantized_weight)

print("Scale search:", scale_result.calibration_errors)
print("Selected alpha:", scale_result.alpha)
print("Clipping search:", clip_result.calibration_errors)
print("Selected clipping ratio:", clip_result.ratio)

for split, inputs in {
    "calibration": calibration_inputs,
    "held-out": heldout_inputs,
}.items():
    reference_outputs = inputs @ projection_weight.T
    for method, outputs in {
        "RTN": inputs @ rtn_weight.T,
        "AWQ scaling": awq_linear(
            inputs, scaled_weight, scale_result.channel_scales,
        ),
        "AWQ scaling + clipping": awq_linear(
            inputs, clipped_weight, scale_result.channel_scales,
        ),
    }.items():
        output_mse = (outputs - reference_outputs).square().mean()
        print(f"{split}, {method}: output MSE = {float(output_mse):.6f}")


### 2.6 Quantization Tradeoffs (6 points)

Consider batch-one autoregressive decoding of **Llama-2-7B** using the weight formats below. The table reports WikiText-2 perplexity; lower is better.

| Method | Weight bits | Group size | Perplexity |
|---|---:|---:|---:|
| FP16 | 16 | — | 5.47 |
| RTN INT4 | 4 | 128 | 5.73 |
| AWQ INT4 | 4 | 128 | 5.60 |
| AWQ INT3 | 3 | 128 | 6.24 |

For this calculation, approximate the model as $N=7.0\times10^9$ weights, all stored in the listed format. Each quantized group stores one 16-bit scale and one $b$-bit zero point in addition to its $b$-bit weight codes. Assume ideal bit packing without padding and AWQ channel scales folded into preceding operators.

Use a memory bandwidth of $1.0\times10^{12}$ bytes/s. Each generated token reads all weights and group metadata once. Ignore other memory traffic and compute/dequantization time. Use decimal GB ($10^9$ bytes).

Source: perplexities are from **Table 4** and the rounded RTX 4090 bandwidth from **Section 4.1** of the [AWQ paper](https://arxiv.org/html/2306.00978v5). The storage and traffic assumptions above define the calculation model.

1. **Calculation (3 points).** For each method, calculate weight storage including group metadata, the bandwidth-limited throughput upper bound in tokens/s, and its ratio to the FP16 bound.
2. **Explanation (3 points).** Explain AWQ INT4 versus RTN INT4 at the same storage cost, the tradeoff when moving from AWQ INT4 to INT3, and why the calculated throughput ratios need not be achieved in a real implementation.


**Answer**

Write your answer here.


### Public Tests — Part 2

Run this cell after running the setup and function definitions for Part 2. Each check reports `ok`, `FAIL`, or `ERROR`; unfinished TODOs report an error. `OK` at the end means all public cases in this part passed.


In [ ]:
# Public tests for Part 2. Run after executing the Part 2 function definitions.
import sys
import unittest


class AWQPublicTests(unittest.TestCase):
    def check_quantized_weight(self, quantized, codes, scales, zero_points, dtype):
        torch.testing.assert_close(quantized.codes, torch.tensor(codes, dtype=torch.long))
        torch.testing.assert_close(
            quantized.scales, torch.tensor(scales, dtype=dtype), rtol=1e-6, atol=0.0,
        )
        torch.testing.assert_close(
            quantized.zero_points, torch.tensor(zero_points, dtype=torch.long),
        )

    def check_error_history(self, history, candidates, expected_errors):
        self.assertEqual(len(history), len(candidates))
        for (candidate, error), expected_candidate, expected_error in zip(
            history, candidates, expected_errors,
        ):
            self.assertIs(type(candidate), float)
            self.assertEqual(candidate, expected_candidate)
            self.assertIs(type(error), float)
            self.assertAlmostEqual(error, expected_error, delta=1e-6)

    def test_q1_group_codes_range_floor_and_dtype(self):
        weight = torch.tensor(
            [[-1.0, 0.5, 0.0, 0.0], [0.2, 0.8, -0.8, -0.2]], dtype=torch.float32,
        )
        original = weight.clone()
        result = quantize_weight_groups(weight, bits=2, group_size=2)
        self.check_quantized_weight(
            result, [[0, 3, 0, 0], [1, 3, 0, 2]],
            [[0.5, 1e-5 / 3], [0.2, 0.2]], [[2, 0], [0, 3]], torch.float32,
        )
        torch.testing.assert_close(weight, original, rtol=0.0, atol=0.0)

    def test_q2_dequantization_of_supplied_codes(self):
        quantized = GroupQuantizedWeight(
            codes=torch.tensor([[0, 2, 3, 1], [1, 2, 0, 3]]),
            scales=torch.tensor([[0.5, 0.25], [2.0, 0.125]]),
            zero_points=torch.tensor([[1, 3], [0, 1]]),
        )
        torch.testing.assert_close(
            dequantize_weight_groups(quantized, group_size=2),
            torch.tensor([[-0.5, 0.5, 0.0, -0.5], [2.0, 4.0, -0.125, 0.25]]),
            check_dtype=False,
        )

    def test_awq1_absolute_means_and_scale_floor(self):
        inputs = torch.tensor([[-6.0, 2.0, 0.0], [2.0, 0.0, 0.0]])
        torch.testing.assert_close(
            activation_channel_scales(inputs, alpha=0.5),
            torch.tensor([141.4213562373095, 70.71067811865476, 0.007071067811865475]),
            rtol=1e-6, atol=1e-8, check_dtype=False,
        )

    def test_awq2_inverse_scaling_on_input_channels(self):
        outputs = awq_linear(
            torch.tensor([[2.0, 1.0, -1.0], [-2.0, 3.0, 2.0]]),
            torch.tensor([[2.0, 2.0, -1.0], [-4.0, 4.0, 1.0]]),
            torch.tensor([2.0, 0.5, 1.0]),
        )
        torch.testing.assert_close(
            outputs, torch.tensor([[7.0, 3.0], [8.0, 30.0]]), check_dtype=False,
        )

    def test_awq3_selects_by_output_mse_and_returns_full_history(self):
        weight = torch.tensor([[-1.0, 0.9375]])
        inputs = torch.tensor([[4.0, 1.0], [-4.0, 1.0]])
        original_weight, original_inputs = weight.clone(), inputs.clone()
        alphas = (1.0, 0.0)
        result = search_awq_scales(weight, inputs, bits=2, group_size=2, alphas=alphas)
        self.assertEqual(result.alpha, 1.0)
        self.check_error_history(result.calibration_errors, alphas, [289 / 288, 833 / 576])
        torch.testing.assert_close(
            result.channel_scales, torch.tensor([2.0, 0.5]), check_dtype=False,
        )
        self.check_quantized_weight(
            result.quantized_weight, [[0, 3]], [[79 / 96]], [[2]], torch.float64,
        )
        torch.testing.assert_close(weight, original_weight, rtol=0.0, atol=0.0)
        torch.testing.assert_close(inputs, original_inputs, rtol=0.0, atol=0.0)

    def test_clip1_clip2_group_bounds_and_unclipped_reference(self):
        weight = torch.tensor([[-2.0, 2.0, 1.0, -1.0]])
        inputs = torch.tensor([[0.25, 1.0, 0.0, 2.0], [0.0, -1.0, 0.0, 1.0]])
        scales = torch.tensor([2.0, 0.5, 2.0, 0.5])
        original_weight, original_inputs, original_scales = (
            weight.clone(), inputs.clone(), scales.clone(),
        )
        ratios = (1.0, 0.5, 0.75)
        result = search_awq_clipping(
            weight, inputs, scales, bits=2, group_size=2, ratios=ratios,
        )
        self.assertEqual(result.ratio, 0.5)
        self.check_error_history(result.calibration_errors, ratios, [577 / 288, 1 / 32, 37 / 72])
        self.check_quantized_weight(
            result.quantized_weight, [[0, 3, 3, 0]], [[1.0, 0.5]], [[2, 1]], torch.float64,
        )
        torch.testing.assert_close(weight, original_weight, rtol=0.0, atol=0.0)
        torch.testing.assert_close(inputs, original_inputs, rtol=0.0, atol=0.0)
        torch.testing.assert_close(scales, original_scales, rtol=0.0, atol=0.0)


unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(AWQPublicTests)
)


## References

The task descriptions provide the information needed to complete the notebook. The sources below describe the full methods and implementations.

- Leviathan, Kalman, and Matias (2023). [Fast Inference from Transformers via Speculative Decoding](https://proceedings.mlr.press/v202/leviathan23a.html). Algorithm 1 and Appendix A.1 cover verification and its distributional guarantee.
- Li et al. (2025). [EAGLE-3: Scaling up Inference Acceleration of Large Language Models via Training-Time Test](https://arxiv.org/html/2503.01840v1). Sections 3.1–3.2 describe feature fusion, recurrence, and training.
- Chen, Liang, and Liu (2026). [DFlash: Block Diffusion for Flash Speculative Decoding](https://arxiv.org/html/2602.06036v1). Section 4.1 describes parallel drafting with target features.
- [DSpark: Confidence-Scheduled Speculative Decoding with Semi-Autoregressive Generation](https://arxiv.org/html/2607.05147v1). Sections 3.1–3.2 describe the drafter and confidence head; Appendix A discusses selection bias.
- [DeepSpec source snapshot](https://github.com/deepseek-ai/DeepSpec/tree/005e03b81cec38b7da6399833d609ee89a2587f2). The notebook uses the `VanillaMarkov` correction and threshold-based prefix selection.
- [SGLang: EAGLE-3 on Vertex](https://www.lmsys.org/blog/2025-12-01-eagle3-vertex/) and [vLLM: Speculative Decoding](https://docs.vllm.ai/en/latest/features/speculative_decoding/). Systems context for decode acceleration.
- Lin et al. (2024). [AWQ: Activation-aware Weight Quantization for On-Device LLM Compression and Acceleration](https://arxiv.org/html/2306.00978v5). Section 3.2 describes scaling and clipping, Section 4.1 analyzes memory bandwidth, and Table 4 reports perplexity.
- MIT Han Lab. [AWQ quantizer](https://github.com/mit-han-lab/llm-awq/blob/main/awq/quantize/quantizer.py), [scale search](https://github.com/mit-han-lab/llm-awq/blob/main/awq/quantize/auto_scale.py), and [clipping search](https://github.com/mit-han-lab/llm-awq/blob/main/awq/quantize/auto_clip.py). Reference implementations for group-wise affine quantization, activation-aware scaling, and calibration-based clipping.
